In [3]:
import math
import numpy as np
import matplotlib.pyplot as plt
import torch
%matplotlib inline

from graphviz import Digraph
from IPython.display import Image, display
from IPython.display import Image, display, HTML

In [4]:
class Value:
  def __init__(self,data, _children=(), _op='', label='') -> None:
    self.data = data
    self.grad = 0.0
    self._prev = set(_children) # this keeps track of the parents of the returned data
    self._op = _op # keeps track of the operations we are using, and so we know what operation we did to the two parents to get the outputed data value
    self.label = label
    self._backward = lambda: None


  def __repr__(self) -> str:  # this allows us to print out a nice looking value and not like bit notation values
    return f"Value(data={self.data})"

  #say we wanna add a and b, we gotta make a function for add since python doesn't know how to add two Value objects
  def __add__(self,other) -> 'Value':
    other = other if isinstance(other, Value) else Value(other)
    out = Value(self.data + other.data, (self, other), '+') # "+" here is addition like normal since we are using the actual data number not the Value object
  
    def _backward(): # has access to self and other due to being insde the add function
      self.grad += 1.0 * out.grad
      other.grad += 1.0 * out.grad
    out._backward = _backward

    return out

  def __mul__(self,other) -> 'Value': # must say "mul" and not "multiply"
    other = other if isinstance(other, Value) else Value(other)
    out = Value(self.data * other.data, (self, other), '*')

    def _backward():
      self.grad += other.data * out.grad
      other.grad += self.data * out.grad
    out._backward = _backward
  
    return out
  
  def __rmul__(self,other) -> 'Value':
    return self * other
  
  def __sub__(self,other) -> 'Value':
    return self + other*-1

  def __pow__(self,other) -> 'Value':
    assert isinstance(other, (float,int))
    out = Value(self.data**other, (self,), '**') # we asserted that other is a plain number for simplicity here. and thus the inputs are self only

    def _backward():
      self.grad += other*(self.data**(other-1)) * out.grad # again we know that other is a plain number. we also ignore the derivative with respect to the exponent bc we would beed a natual log. which is icky ig
    out._backward = _backward
    
    return out

  def __truediv__(self,other) -> 'Value':
    # out = Value(self.data/other.data, (self, other), '/') # we dont need to define anything new since we have * and **
    out = self * other**-1 # we dont need .data here since we are using operations we already define

    # do not need any backward code here bc we are using * and ** which we already defined

    return out

    
  def exp(self) -> 'Value': # this defines just e^x, this isnt general exponentition 
    out = Value(math.exp(self.data), (self,), 'exp')

    def _backward():
      self.grad += out.data * out.grad #the derivative of out.data (e^x) is e^x so the local derivative is the same as the output data
    out._backward = _backward

    return out

  def tanh(self) -> 'Value':
    x = self.data
    t = (math.exp(2*x) - 1)/(math.exp(2*x) + 1)
    out = Value(t, (self, ), 'tanh')

    def _backward():
      self.grad += (1 - t**2)  * out.grad # local derivative * out.grad
    out._backward = _backward
  
    return out
  
  def backward(self):
    topo = [] # unsure why it is called topo and why we keep track of it honestly ****
    visited = set() # set visited we use to keep track of nodes we have already iterated over
    def build_topo(v): # function with arbitrary input v
      if v not in visited: # if v isnt visted yet, call the bellow on that node. if it is already visited, then move on to the next node
        visited.add(v) # count it as visited so we can keep track of which nodes aren't and are visited
        for child in v._prev: # checks if that node as children
          build_topo(child) # if so, calls the build topo function on that child(s)
        topo.append(v) # if there are no more children to call to, append that node to topo ****
    build_topo(self) #call build topo on that output of the neuron we want to interate backwards on

    self.grad = 1.0
    for node in reversed(topo):
      node._backward()


In [5]:
!apt-get install -y graphviz

from graphviz import Digraph
from IPython.display import Image, display

def trace(root):
    nodes, edges = set(), set()
    def build(v):
        if v not in nodes:
            nodes.add(v)
            for child in v._prev:
                edges.add((child, v))
                build(child)
    build(root)
    return nodes, edges

def draw_dot(root):
    dot = Digraph(format='png', graph_attr={'rankdir': 'LR'})
    nodes, edges = trace(root)
    for n in nodes:
        uid = str(id(n))
        dot.node(name=uid, label=f"{{ {n.label} | data {n.data:.4f} | grad {n.grad:.4f} }}", shape='record')
        if n._op:
            dot.node(name=uid + n._op, label=n._op)
            dot.edge(uid + n._op, uid)
    for n1, n2 in edges:
        dot.edge(str(id(n1)), str(id(n2)) + n2._op)
    return dot


E: Could not open lock file /var/lib/dpkg/lock-frontend - open (13: Permission denied)
E: Unable to acquire the dpkg frontend lock (/var/lib/dpkg/lock-frontend), are you root?


In [6]:
#inputs x1 and x2
x1 = Value(2.0, label='x1')
x2 = Value(1.0, label='x2')

#weights w1 and w2
w1 = Value(-3.0, label='w1')
w2 = Value(1.0, label='w2')

#bias of the neuron
b = Value(6.7, label='b')

#x1*w1 + x2*w2 + b
x1w1 = x1*w1; x1w1.label='x1*w1'
x2w2 = x2*w2; x2w2.label = 'x2*w2'
x1w1x2w2 = x1w1 + x2w2; x1w1x2w2.label = 'x1*w1 + x2*w2'
n = x1w1x2w2 + b; n.label = 'n'
# o = n.tanh()


e = (2*n).exp()
o = (e - 1)/(e + 1)

o.label = 'o'
o.backward()

diagram = draw_dot(o)
svg = diagram.pipe(format='svg').decode('utf-8')
display(HTML(f'<div style="overflow:auto; height:500px; background:white">{svg}</div>'))

In [7]:
x1 = torch.Tensor([2.0]).double() ; x1.requires_grad = True
x2 = torch.Tensor([0.0]).double() ; x2.requires_grad = True
w1 = torch.Tensor([-3.0]).double() ; w1.requires_grad = True
w2 = torch.Tensor([1.0]).double() ; w2.requires_grad = True
b = torch.Tensor([6.88]).double() ; b.requires_grad = True
n = w1*x1 + w2*x2 + b
o = torch.tanh(n)

print(o.item())
o.backward()

print('x1', x1.grad.item())
print('x2', x2.grad.item())
print('w1', w1.grad.item())
print('w2', w2.grad.item())

0.7064193777288968
x1 -1.5029149883073547
x2 0.5009716627691182
w1 1.0019433255382364
w2 0.0


In [ ]:
class Neuron:

    def __init__(self, nin): # takes the input and the number of inputs
        self.w = [Value(random.uniform(-1,1) for _ in range(nin))]
        self.b = Value(random.uniform(-1,1))

    def __call__(self, x): # call the neuron function with the inputed x values
        act = sum((w1*x1 for w1,x1 in zip(self.w,x)), self.b) # the function multiplies the weight and adds the bias
        out = act.tanh()

SyntaxError: invalid syntax (1590701055.py, line 1)